<a href="https://colab.research.google.com/github/ddesiccantt/EECS498-DC/blob/main/A2_Q1_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn.functional as F

# 固定随机种子，方便重复实验
torch.manual_seed(0)


# ===== 1. 准备数据 =====

# [N, D]：两个样本，每个样本两个特征
X = torch.tensor([
    [1., 2.],
    [3., 4.]
])

# [N]：每个样本的真实类别
y = torch.tensor([0, 2])

# [D, C]：每列对应一个类别的权重
W = torch.tensor([
    [1., 0., -1.],
    [0., 1.,  1.]
])


# ===== 2. Softmax loss 和手动梯度 =====

def softmax_loss_and_gradient(X, W, y, reg=0.01):
    N = X.shape[0]
    rows = torch.arange(N, device=X.device)

    # 分类分数：[N, C]
    scores = X @ W

    # 每行减去最大值，防止指数溢出
    shifted_scores = scores - scores.max(
        dim=1, keepdim=True
    ).values

    # 转换为概率
    exp_scores = shifted_scores.exp()
    sum_exp = exp_scores.sum(dim=1, keepdim=True)
    probs = exp_scores / sum_exp

    # 稳定地计算 log 概率
    log_probs = shifted_scores - sum_exp.log()

    # 分类损失 + L2 正则化损失
    data_loss = -log_probs[rows, y].mean()
    reg_loss = reg * (W ** 2).sum()
    loss = data_loss + reg_loss

    # 分类损失对分数的梯度
    dscores = probs.clone()
    dscores[rows, y] -= 1
    dscores /= N

    # 总损失对权重的梯度
    dW = X.T @ dscores
    dW += 2 * reg * W

    return loss, dW


# ===== 3. 用自动求导检查手动梯度 =====

reg = 0.01

_, manual_dW = softmax_loss_and_gradient(X, W, y, reg)

# 创建独立的权重副本，开启自动求导
W_check = W.detach().clone().requires_grad_(True)

# cross_entropy 接收原始分数
loss_check = (
    F.cross_entropy(X @ W_check, y)
    + reg * (W_check ** 2).sum()
)

loss_check.backward()
auto_dW = W_check.grad

print("手动梯度：")
print(manual_dW)

print("自动梯度：")
print(auto_dW)

max_error = (manual_dW - auto_dW).abs().max().item()
gradients_match = torch.allclose(
    manual_dW, auto_dW, atol=1e-6, rtol=1e-4
)

print("最大绝对误差：", max_error)
print("梯度是否接近：", gradients_match)


# ===== 4. 小批量 SGD 训练 =====

learning_rate = 0.1
num_steps = 1000

# 目前只有两个样本，每次随机抽一个
batch_size = 1
N = X.shape[0]

for step in range(num_steps):
    # 随机抽样：索引范围是 0 到 N-1
    indices = torch.randint(
        N, (batch_size,), device=X.device
    )

    X_batch = X[indices]
    y_batch = y[indices]

    # 计算本批次的 loss 和梯度
    batch_loss, dW = softmax_loss_and_gradient(
        X_batch, W, y_batch, reg
    )

    # 更新权重
    W = W - learning_rate * dW

    # 用全部训练数据计算 loss，方便比较训练进展
    if (step + 1) % 100 == 0:
        train_loss, _ = softmax_loss_and_gradient(
            X, W, y, reg
        )
        print(
            f"更新 {step + 1} 次后，"
            f"训练总 loss = {train_loss.item():.4f}"
        )


# ===== 5. 查看训练结果 =====

final_loss, _ = softmax_loss_and_gradient(X, W, y, reg)

final_scores = X @ W
final_probs = torch.softmax(final_scores, dim=1)
predictions = final_scores.argmax(dim=1)

accuracy = (predictions == y).float().mean()

print("\n最终总 loss：", final_loss.item())

print("预测概率：")
print(final_probs)

print("预测类别：", predictions)
print("真实类别：", y)
print(f"训练准确率：{accuracy.item():.1%}")

print("训练后的权重：")
print(W)

手动梯度：
tensor([[ 0.0152,  1.3461, -1.3614],
        [-0.2691,  2.0069, -1.6978]])
自动梯度：
tensor([[ 0.0152,  1.3461, -1.3614],
        [-0.2691,  2.0069, -1.6978]])
最大绝对误差： 0.0
梯度是否接近： True
更新 100 次后，训练总 loss = 0.6145
更新 200 次后，训练总 loss = 0.4924
更新 300 次后，训练总 loss = 0.4351
更新 400 次后，训练总 loss = 0.3826
更新 500 次后，训练总 loss = 0.3601
更新 600 次后，训练总 loss = 0.3641
更新 700 次后，训练总 loss = 0.3579
更新 800 次后，训练总 loss = 0.3809
更新 900 次后，训练总 loss = 0.3607
更新 1000 次后，训练总 loss = 0.3542

最终总 loss： 0.35421913862228394
预测概率：
tensor([[7.6176e-01, 1.5002e-02, 2.2324e-01],
        [1.2377e-01, 2.1381e-04, 8.7601e-01]])
预测类别： tensor([0, 2])
真实类别： tensor([0, 2])
训练准确率：100.0%
训练后的权重：
tensor([[-1.9685, -0.4747,  2.4432],
        [ 1.9334, -0.7772, -0.8861]])


In [3]:
torch.manual_seed(0)

# 三个类别，每类 100 个样本
# 每个样本仍然只有两个特征
X0 = torch.randn(100, 2) * 0.5 + torch.tensor([2., 0.])
X1 = torch.randn(100, 2) * 0.5 + torch.tensor([0., 2.])
X2 = torch.randn(100, 2) * 0.5 + torch.tensor([-2., -2.])

X_all = torch.cat([X0, X1, X2], dim=0)

y_all = torch.cat([
    torch.full((100,), 0, dtype=torch.long),
    torch.full((100,), 1, dtype=torch.long),
    torch.full((100,), 2, dtype=torch.long)
])

# 随机打乱 300 个样本的索引
indices = torch.randperm(300)

# 240 个用于训练，60 个用于验证
train_indices = indices[:240]
val_indices = indices[240:]

X_train = X_all[train_indices]
y_train = y_all[train_indices]

X_val = X_all[val_indices]
y_val = y_all[val_indices]

print(X_train.shape)  # torch.Size([240, 2])
print(X_val.shape)    # torch.Size([60, 2])

# ===== 2. Softmax loss 和手动梯度 =====

def softmax_loss_and_gradient(X, W, y, reg=0.01):
    N = X.shape[0]
    rows = torch.arange(N, device=X.device)

    # 分类分数：[N, C]
    scores = X @ W

    # 每行减去最大值，防止指数溢出
    shifted_scores = scores - scores.max(
        dim=1, keepdim=True
    ).values

    # 转换为概率
    exp_scores = shifted_scores.exp()
    sum_exp = exp_scores.sum(dim=1, keepdim=True)
    probs = exp_scores / sum_exp

    # 稳定地计算 log 概率
    log_probs = shifted_scores - sum_exp.log()

    # 分类损失 + L2 正则化损失
    data_loss = -log_probs[rows, y].mean()
    reg_loss = reg * (W ** 2).sum()
    loss = data_loss + reg_loss

    # 分类损失对分数的梯度
    dscores = probs.clone()
    dscores[rows, y] -= 1
    dscores /= N

    # 总损失对权重的梯度
    dW = X.T @ dscores
    dW += 2 * reg * W

    return loss, dW

# 两个特征、三个类别，随机初始化较小的权重
W = torch.randn(2, 3) * 0.01

learning_rate = 0.1
reg = 0.01
batch_size = 32
num_steps = 500

N = X_train.shape[0]

for step in range(num_steps):
    # 只从训练集中抽取样本
    indices = torch.randint(
        N, (batch_size,), device=X_train.device
    )

    X_batch = X_train[indices]
    y_batch = y_train[indices]

    loss, dW = softmax_loss_and_gradient(
        X_batch, W, y_batch, reg
    )

    W = W - learning_rate * dW

    # 每更新 100 次，检查训练集和验证集
    if (step + 1) % 100 == 0:
        train_predictions = (X_train @ W).argmax(dim=1)
        val_predictions = (X_val @ W).argmax(dim=1)

        train_accuracy = (
            train_predictions == y_train
        ).float().mean()

        val_accuracy = (
            val_predictions == y_val
        ).float().mean()

        print(
            f"更新 {step + 1} 次 | "
            f"训练准确率：{train_accuracy.item():.1%} | "
            f"验证准确率：{val_accuracy.item():.1%}"
        )

torch.Size([240, 2])
torch.Size([60, 2])
更新 100 次 | 训练准确率：99.6% | 验证准确率：100.0%
更新 200 次 | 训练准确率：99.6% | 验证准确率：100.0%
更新 300 次 | 训练准确率：99.6% | 验证准确率：100.0%
更新 400 次 | 训练准确率：99.6% | 验证准确率：100.0%
更新 500 次 | 训练准确率：99.6% | 验证准确率：100.0%


In [5]:
import torch
import torch.nn.functional as F


# ===== 1. 生成练习数据 =====

torch.manual_seed(0)

# 三个类别，每类 100 个样本，每个样本两个特征
X0 = torch.randn(100, 2) * 0.5 + torch.tensor([2., 0.])
X1 = torch.randn(100, 2) * 0.5 + torch.tensor([0., 2.])
X2 = torch.randn(100, 2) * 0.5 + torch.tensor([-2., -2.])

X_all = torch.cat([X0, X1, X2], dim=0)

y_all = torch.cat([
    torch.full((100,), 0, dtype=torch.long),
    torch.full((100,), 1, dtype=torch.long),
    torch.full((100,), 2, dtype=torch.long)
])


# ===== 2. 划分训练集和验证集 =====

indices = torch.randperm(X_all.shape[0])

train_indices = indices[:240]
val_indices = indices[240:]

X_train = X_all[train_indices]
y_train = y_all[train_indices]

X_val = X_all[val_indices]
y_val = y_all[val_indices]

print("训练集形状：", X_train.shape)
print("验证集形状：", X_val.shape)


# ===== 3. Softmax loss 和手动梯度 =====

def softmax_loss_and_gradient(X, W, y, reg=0.01):
    N = X.shape[0]
    rows = torch.arange(N, device=X.device)

    # 分类分数：[N, C]
    scores = X @ W

    # 每行减去最大值，防止指数溢出
    shifted_scores = scores - scores.max(
        dim=1, keepdim=True
    ).values

    # Softmax 概率
    exp_scores = shifted_scores.exp()
    sum_exp = exp_scores.sum(dim=1, keepdim=True)
    probs = exp_scores / sum_exp

    # 稳定地计算 log 概率
    log_probs = shifted_scores - sum_exp.log()

    # 平均分类损失 + L2 正则化损失
    data_loss = -log_probs[rows, y].mean()
    reg_loss = reg * (W ** 2).sum()
    loss = data_loss + reg_loss

    # 分类损失对分数的梯度
    dscores = probs.clone()
    dscores[rows, y] -= 1
    dscores /= N

    # 总损失对权重的梯度
    dW = X.T @ dscores
    dW += 2 * reg * W

    return loss, dW


# ===== 4. 初始化权重 =====

torch.manual_seed(0)

# 两个特征，三个类别
initial_W = torch.randn(2, 3) * 0.01


# ===== 5. 用自动求导检查手动梯度 =====

check_reg = 0.01

# 用少量训练样本进行检查
X_check = X_train[:10]
y_check = y_train[:10]

_, manual_dW = softmax_loss_and_gradient(
    X_check, initial_W, y_check, check_reg
)

W_check = initial_W.detach().clone().requires_grad_(True)

# cross_entropy 直接接收原始分类分数
loss_check = (
    F.cross_entropy(X_check @ W_check, y_check)
    + check_reg * (W_check ** 2).sum()
)

loss_check.backward()
auto_dW = W_check.grad

max_error = (manual_dW - auto_dW).abs().max().item()

gradients_match = torch.allclose(
    manual_dW, auto_dW,
    atol=1e-6, rtol=1e-4
)

print("\n梯度最大绝对误差：", max_error)
print("梯度检查是否通过：", gradients_match)


# ===== 6. 比较不同超参数 =====

learning_rates = [0.01, 0.1, 1.0]
reg_values = [0.0, 0.01]

batch_size = 32
num_steps = 500
N = X_train.shape[0]

best_accuracy = -1.0
best_W = None
best_params = None

for lr in learning_rates:
    for reg in reg_values:
        # 每组实验从相同的权重开始
        W = initial_W.clone()

        # 各组使用相同的随机抽样序列
        torch.manual_seed(1)

        for step in range(num_steps):
            # 只从训练集中随机抽样
            batch_indices = torch.randint(
                N, (batch_size,),
                device=X_train.device
            )

            X_batch = X_train[batch_indices]
            y_batch = y_train[batch_indices]

            # 计算 loss 和权重梯度
            loss, dW = softmax_loss_and_gradient(
                X_batch, W, y_batch, reg
            )

            # SGD 更新权重
            W = W - lr * dW

        # 验证集只用于评估
        val_predictions = (X_val @ W).argmax(dim=1)

        val_accuracy = (
            val_predictions == y_val
        ).float().mean().item()

        print(
            f"\nlr={lr}, reg={reg} | "
            f"验证准确率：{val_accuracy:.1%}"
        )

        # 保存验证准确率最高的模型
        if val_accuracy > best_accuracy:
            best_accuracy = val_accuracy
            best_W = W.clone()
            best_params = (lr, reg)


# ===== 7. 查看最佳模型 =====

best_lr, best_reg = best_params

train_predictions = (X_train @ best_W).argmax(dim=1)
val_predictions = (X_val @ best_W).argmax(dim=1)

train_accuracy = (
    train_predictions == y_train
).float().mean().item()

val_accuracy = (
    val_predictions == y_val
).float().mean().item()

print("\n最佳学习率：", best_lr)
print("最佳正则化强度：", best_reg)
print(f"训练准确率：{train_accuracy:.1%}")
print(f"验证准确率：{val_accuracy:.1%}")

print("\n前 10 个验证样本的预测：")
print(val_predictions[:10])

print("对应的真实类别：")
print(y_val[:10])

print("\n最佳权重：")
print(best_W)

# 生成独立测试数据，每类 30 个样本
torch.manual_seed(42)

X_test0 = torch.randn(30, 2) * 0.5 + torch.tensor([2., 0.])
X_test1 = torch.randn(30, 2) * 0.5 + torch.tensor([0., 2.])
X_test2 = torch.randn(30, 2) * 0.5 + torch.tensor([-2., -2.])

X_test = torch.cat([X_test0, X_test1, X_test2], dim=0)

y_test = torch.tensor(
    [0] * 30 + [1] * 30 + [2] * 30,
    dtype=torch.long
)

# 直接使用已经选好的权重
test_predictions = (X_test @ best_W).argmax(dim=1)

test_accuracy = (
    test_predictions == y_test
).float().mean().item()

print(f"测试准确率：{test_accuracy:.1%}")

训练集形状： torch.Size([240, 2])
验证集形状： torch.Size([60, 2])

梯度最大绝对误差： 1.1920928955078125e-07
梯度检查是否通过： True

lr=0.01, reg=0.0 | 验证准确率：100.0%

lr=0.01, reg=0.01 | 验证准确率：100.0%

lr=0.1, reg=0.0 | 验证准确率：100.0%

lr=0.1, reg=0.01 | 验证准确率：100.0%

lr=1.0, reg=0.0 | 验证准确率：100.0%

lr=1.0, reg=0.01 | 验证准确率：100.0%

最佳学习率： 0.01
最佳正则化强度： 0.0
训练准确率：99.6%
验证准确率：100.0%

前 10 个验证样本的预测：
tensor([1, 2, 0, 1, 1, 1, 0, 1, 2, 1])
对应的真实类别：
tensor([1, 2, 0, 1, 1, 1, 0, 1, 2, 1])

最佳权重：
tensor([[ 0.9909, -0.2803, -0.7199],
        [-0.2929,  0.9884, -0.7147]])
测试准确率：100.0%
